# Daily Challenge: LangChain Pipelines with Open-Source LLMs (Student)
Use this guided notebook with TODOs. Runs on CPU with small HF models (e.g., flan-t5-small).

## What you'll learn
- Set up LangChain with lightweight open-source models.
- Build an LLMChain using a prompt template.
- Compose a two-step Runnable pipeline (summary ? bullets).
- Bonus: add a simple conversation chain with memory.

## What you will create
- Installed environment for LangChain + transformers.
- LLMChain that rewrites text in a simpler style.
- Runnable pipeline that summarizes then bullet-izes text.
- (Bonus) Conversation chain showing memory.

## Part 1: Environment setup (fast)
Install needed packages. CPU is fine for tiny models.

In [3]:

# TODO: verify hardware (optional)
!nvidia-smi || echo "CPU runtime"


/bin/bash: line 1: nvidia-smi: command not found
CPU runtime


In [5]:

# TODO: install dependencies
!pip install "transformers==4.37.2" "langchain==0.1.7" "langchain-community==0.0.20" "langchain-core==0.1.23"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.9/815.9 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.2/241.2 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing 

## Part 2: Load a tiny model and build your first LLMChain
Use a small model (e.g., google/flan-t5-small) to keep inference quick.

In [1]:
# Importation des bibliothèques nécessaires
# transformers : pour charger le modèle localement
# langchain_community : pour intégrer le modèle HuggingFace dans LangChain
# langchain : composants de base pour les prompts et les chaînes
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain

In [2]:
# Sélection d'un modèle léger (Flan-T5-Small)
# Ce modèle est idéal pour les tests sur CPU car il consomme peu de mémoire
model_name = "google/flan-t5-small"

In [3]:
# Chargement du Tokenizer (pour transformer le texte en nombres)
# et du Modèle (le cerveau qui génère le texte)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [4]:
# Création d'un pipeline HuggingFace standard
gen_pipeline = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
)

# Encapsulation du pipeline dans un objet LangChain pour l'utiliser avec ses outils
llm = HuggingFacePipeline(pipeline=gen_pipeline)

In [5]:
# Définition d'un PromptTemplate : c'est un moule avec une variable {text}
template = "Rewrite this text to be simpler for beginners:{text}"
prompt = PromptTemplate(template=template, input_variables=["text"])

# Création d'une LLMChain : elle lie le Prompt et le LLM ensemble
chain = LLMChain(prompt=prompt, llm=llm)

# Exécution de la chaîne avec un exemple de texte
sample_text = "LangChain helps you build LLM apps by composing prompts, models, and tools."
rewritten = chain.run(text=sample_text)
print(rewritten)

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The function `run` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(


LLM apps are built using a lma file.


## Part 3: Two-step pipeline (summary ? bullets)
Summarize a paragraph, then turn it into 3 bullets using the same LLM.

In [6]:
# Définition de deux modèles de prompts distincts pour notre pipeline en deux étapes
# Étape A : Résumer le paragraphe
summary_prompt = PromptTemplate(
    template="Summarize the following text briefly:\n\n{paragraph}",
    input_variables=["paragraph"],
)

# Étape B : Transformer le résultat précédent en liste à puces
bullets_prompt = PromptTemplate(
    template="Turn this summary into 3 bullet points:\n\n{summary}",
    input_variables=["summary"],
)

In [7]:
# Utilisation de LCEL (LangChain Expression Language) avec l'opérateur pipe '|'
# On définit la première étape
summary_chain = summary_prompt | llm

# On compose le pipeline complet :
# 1. On injecte le paragraphe
# 2. La sortie de summary_chain devient l'entrée 'summary' pour l'étape suivante
# 3. On applique le prompt des puces puis le LLM
summarize_then_bullets = (
    {"summary": summary_chain}
    | bullets_prompt
    | llm
)

In [8]:
paragraph = """LangChain is a framework for building applications with large language models by composing prompts, models, and tools. It supports chains, agents, and retrieval workflows."""
bullets_output = summarize_then_bullets.invoke({"paragraph": paragraph})
print(bullets_output)


The LangChain framework is a framework for building applications with large language models by composing prompts, models, and tools.


## Part 4 (Bonus): Conversation chain with memory
Show how two turns keep context.

In [9]:
# Mise en place d'une chaîne de conversation avec mémoire
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

# Le BufferMemory stocke l'historique des échanges pour que le modèle garde le contexte
memory = ConversationBufferMemory()
convo = ConversationChain(llm=llm, memory=memory, verbose=False)

# Test de la mémoire sur deux tours de parole
reply1 = convo.predict(input="Hi there! What's LangChain?")
reply2 = convo.predict(input="Can it help me build a simple chatbot?")
print("Turn 1:", reply1)
print("Turn 2:", reply2)

Turn 1: LangChain is a fictional character in the "LangChain" series.
Turn 2: It's a simple chatbot.


## Your observations
- **Latency**: Low/Fast. Using a small model like `flan-t5-small` allows for quick inference even on a CPU runtime.
- **Quality**: Low. The model provides very brief summaries and sometimes fails to follow complex instructions (like creating exactly 3 bullet points).
- **Quirks**: High hallucination rate. For instance, the model incorrectly identified LangChain as a 'fictional character' in the conversation turn.